In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport

from bs4 import BeautifulSoup
import requests
from io import StringIO
import time
import unicodedata

In [5]:
url = 'https://arp.sn/bases-de-donnees/medicaments-autorises'
reponse = requests.get(url)
soup = BeautifulSoup(reponse.text, 'html.parser')
base_url = "https://arp.sn/bases-de-donnees/medicaments-autorises"
all_data = []
total_pages = 299

# En-têtes réels identifiés dans la structure
columns_names = [
    "Nom du Médicament",
    "DCI",
    "Dosage",
    "Présentation",
    "Forme Galénique",
    "Voie d'administration",
    "Laboratoire",
    "Classe Thérapeutique"
]

print("Début de l'extraction de toutes les pages d'AMM...")

for page in range(1, total_pages + 1):
    url = f"{base_url}?page={page}"
    try:
        response = requests.get(url, timeout=15)
        if response.status_code == 200:
            # Lecture des tables de la page
            html_io = StringIO(response.text)
            tables = pd.read_html(html_io)

            # On filtre pour trouver la table contenant les médicaments
            # La table cible a généralement 8 colonnes
            page_added = False
            for df in tables:
                if df.shape[1] == 8 and not df.empty:
                    # Si la première ligne contient les noms de colonnes ou des valeurs vides, on les nettoie si besoin
                    df.columns = columns_names
                    all_data.append(df)
                    page_added = True

            if page_added:
                print(f"Page {page}/{total_pages} récupérée avec succès.", end='\r')
            else:
                print(f"Page {page}/{total_pages} : Aucune table de données valide trouvée.")
        else:
            print(f"Page {page}/{total_pages} : Erreur HTTP {response.status_code}")
    except Exception as e:
        print(f"Page {page}/{total_pages} : Erreur détectée -> {str(e)}")

    # Un court délai pour ne pas surcharger le serveur
    time.sleep(0.5)

Début de l'extraction de toutes les pages d'AMM...


In [8]:
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df = final_df.dropna(how='all')

    output_file = '../data/raw/medicaments_amm_senegal.csv'
    final_df.to_csv(output_file, index=False)
    print(f"\nExtraction terminée ! {len(final_df)} médicaments extraits et enregistrés dans '{output_file}'.")
    display(final_df.head())
else:
    print("\nAucune donnée n'a pu être extraite.")

df = pd.read_csv("../data/raw/medicaments_amm_senegal.csv")

df.dropna(inplace=True)

df.drop_duplicates(inplace=True)

ProfileReport(df)


Extraction terminée ! 7455 médicaments extraits et enregistrés dans '../data/raw/medicaments_amm_senegal.csv'.


,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Voie d'administration,Laboratoire,Classe Thérapeutique
0,5 DF,CHLORURE DE SODIUM,5 g,FL/100ml,solution pour perfusion,intraveineuse,ACULIFE HEALTHCARE PVT,SOLUTION POUR REHYDRATATION
1,5-FLUOROURACILE SANDOZ,5-FLUOROURACILE,250 mg / 5 ml,FL/5ml,solution injectable,injectable,SANDOZ,ANTICANCEREUX
2,5-FLUOROURACILE SANDOZ,5-FLUOROURACILE,1000 mg / 20 ml,FL/20ml,solution injectable,intraveineuse,SANDOZ,ANTICANCEREUX
3,A-CERUMEN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ABACAVIR,ABACAVIR,300 mg,B/60,comprimé,orale,HETERO,ANTIRETROVIRAL


Render HTML: 100%|██████████| 1/1 [00:00<00:00,  2.90it/s]


In [9]:
def normalize_text(text):
    if not isinstance(text, str):
        return text
    text = unicodedata.normalize('NFD', text)
    text = ''.join([c for c in text if unicodedata.category(c) != 'Mn'])
    return text.lower().strip()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].apply(normalize_text)

display(df.head())
print(f"Dimensions actuelles du dataset : {df.shape}")

,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Voie d'administration,Laboratoire,Classe Thérapeutique
0,5 df,chlorure de sodium,5 g,fl/100ml,solution pour perfusion,intraveineuse,aculife healthcare pvt,solution pour rehydratation
1,5-fluorouracile sandoz,5-fluorouracile,250 mg / 5 ml,fl/5ml,solution injectable,injectable,sandoz,anticancereux
2,5-fluorouracile sandoz,5-fluorouracile,1000 mg / 20 ml,fl/20ml,solution injectable,intraveineuse,sandoz,anticancereux
4,abacavir,abacavir,300 mg,b/60,comprime,orale,hetero,antiretroviral
5,abacavir / lamivudine,abacavir / lamivudine,600 mg / 300mg,b/60,comprime dispersible,orale,cipla,antiviral


Dimensions actuelles du dataset : (7363, 8)


In [10]:
# Nombre de valeurs uniques par colonne
for col in df.columns:
    print(f"{col} : {df[col].nunique()} valeurs uniques")

from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()
label_encoders = {}

for col in df_encoded.columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

display(df_encoded.head())

Nom du Médicament : 4886 valeurs uniques
DCI : 2609 valeurs uniques
Dosage : 2123 valeurs uniques
Présentation : 664 valeurs uniques
Forme Galénique : 321 valeurs uniques
Voie d'administration : 133 valeurs uniques
Laboratoire : 1342 valeurs uniques
Classe Thérapeutique : 690 valeurs uniques


,Nom du Médicament,DCI,Dosage,Présentation,Forme Galénique,Voie d'administration,Laboratoire,Classe Thérapeutique
0,0,883,1666,427,284,61,28,635
1,1,8,1287,530,263,36,1082,214
2,1,8,659,476,263,61,1082,214
4,2,11,1408,263,38,95,564,321
5,3,13,1974,263,50,95,312,358
